In [51]:
!pip3 install langchain crewai langchain-community langchain-openai duckduckgo-search crewai-tools databricks-sdk pydantic-ai --break-system-packages

In [52]:
!wget "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/3xGOgzMOv5jhRsA3A8N9fQ/leftover.py"

--2026-03-14 23:35:21--  https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/3xGOgzMOv5jhRsA3A8N9fQ/leftover.py
Resolving cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)... 169.63.118.104
Connecting to cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)|169.63.118.104|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 588 [application/x-python]
Saving to: ‘leftover.py.13’

leftover.py.13      100%[===================>]     588  --.-KB/s    in 0s      

2026-03-14 23:35:23 (112 MB/s) - ‘leftover.py.13’ saved [588/588]



In [ ]:
import sys
sys.path.append(".")

from leftover import LeftoversCrew
from dotenv import load_dotenv

load_dotenv()
import os

files = os.listdir('.')
print(files)

from pydantic import BaseModel, Field
from typing import List, Dict, Optional
from IPython.display import display, JSON, Markdown
from datetime import datetime
import re, json

['01a-Lab.ipynb', 'meals.json', '02b-Lab.ipynb', 'leftover.py.12', 'leftover.py.3', 'leftover.py.4', 'leftover.py', '__pycache__', 'leftover.py.5', 'leftover.py.2', 'leftover.py.13', '01b-Lab.ipynb', '.env', '02a-Lab.ipynb', 'leftover.py.9', 'leftover.py.11', 'leftover.py.7', 'leftover.py.1', 'leftover.py.6', 'leftover.py.10', 'leftover.py.8']


### Creating the Grocery Shopping Assistant Structure

We start by defining Pydantic classes to keep grocery data structured, consistent, and useful.  
Instead of vague text like “get some chicken,” models enforce clear fields (for example: item name, quantity, price, section), so agents produce reliable outputs like “2 lbs chicken breast, $8–12, meat section.”

Using `BaseModel`, `Field`, and `List`, we can validate required fields and build nested shopping structures (such as categories containing multiple grocery items).

##### Our Grocery Shopping Data Structure

1. ```GroceryItem```         → Individual grocery item with details
2. ```MealPlan```           → Recipe information with researched ingredients  
3. ```ShoppingCategory```   → Store section with organized items
4. ```GroceryShoppingPlan``` → Complete shopping strategy with budget analysis


##### `GroceryItem`

Think of this as designing the shopping list template before going to the store. BaseModel gives us the foundation, Field decorates our items with clear descriptions. This structure helps both humans and AI understand grocery data consistently, and includes several key string inputs:

- **`name`** – The specific grocery item to purchase (for example, "Chicken Breast").
- **`quantity`** – How much to buy in clear measurements (for example, "2 lbs", "1 gallon").
- **`estimated_price`** – The expected cost range for budgeting (for example, "$8-12").
- **`category`** – Which store section to find it in (for example, "Meat", "Produce", "Dairy").


In [54]:
class GroceryItem(BaseModel):
    """Individual grocery item"""
    name: str = Field(description="Name of the grocery item")
    quantity: str = Field(description="Quantity needed (for example, '2 lbs', '1 gallon')")
    estimated_price: str = Field(description="Estimated price (for example, '$3-5')")
    category: str = Field(description="Store section (for example, 'Produce', 'Dairy')")

In [55]:
sample_item = GroceryItem(
    name="Chicken Breast",
    quantity="2 lbs",
    estimated_price="$8-12",
    category="Meat"
)
sample_item

GroceryItem(name='Chicken Breast', quantity='2 lbs', estimated_price='$8-12', category='Meat')

In [56]:
# Display structured data
print("🛒 Sample Grocery Item Structure:")
print(type(sample_item))
display(JSON(sample_item.model_dump()))

🛒 Sample Grocery Item Structure:
<class '__main__.GroceryItem'>


<IPython.core.display.JSON object>

This class lets us **create multiple grocery items**. Here, we're generating a **sample grocery item for chicken breast** so we can demonstrate how individual items are structured.

You can create multiple grocery items to build complete shopping lists:
- The first might cover proteins - essential ingredients for main dishes
- The second could focus on vegetables that provide nutrition and flavor  
- A third might include pantry staples such as rice or cooking oils


##### `MealPlan`

It represents a complete meal with all the details needed for cooking, including:

- **`meal_name`** – The name of the dish being prepared (for example, "Chicken Stir Fry").
- **`difficulty_level`** – How challenging it is to cook ("Easy", "Medium", "Hard").
- **`servings`** – Number of people the meal will feed (integer value).
- **`researched_ingredients`** – List of ingredients found through AI research (note this is a list).


In [ ]:
class MealPlan(BaseModel):
    """Simple meal plan"""
    meal_name: str = Field(description="Name of the meal")
    difficulty_level: str = Field(description="'Easy', 'Medium', 'Hard'")
    servings: int = Field(description="Number of people it serves")
    researched_ingredients: List[str] = Field(description="Ingredients found through research")

In [58]:
sample_meal = MealPlan(
    meal_name="Chicken Stir Fry",
    difficulty_level="Easy",
    servings=4,
    researched_ingredients=["chicken breast", "broccoli", "bell peppers", "garlic", "soy sauce", "rice"]
)
sample_meal

MealPlan(meal_name='Chicken Stir Fry', difficulty_level='Easy', servings=4, researched_ingredients=['chicken breast', 'broccoli', 'bell peppers', 'garlic', 'soy sauce', 'rice'])

In [59]:
print("\n🍽️ Sample Meal Plan Structure:")
display(JSON(sample_meal.model_dump()))


🍽️ Sample Meal Plan Structure:


<IPython.core.display.JSON object>

##### `ShoppingCategory`

It organizes items by store layout for efficient shopping, containing:

- **`section_name`** – The store department name (for example, "Produce", "Meat & Poultry").
- **`items`** – A collection of GroceryItem objects in this section (note this is a list).
- **`estimated_total`** – The expected cost for all items in this category.


In [60]:
class ShoppingCategory(BaseModel):
    """Store section with items"""
    section_name: str = Field(description="Store section (for example, 'Produce', 'Dairy')")
    items: List[GroceryItem] = Field(description="Items in this section")
    estimated_total: str = Field(description="Estimated cost for this section")

In [61]:
sample_section = ShoppingCategory(
    section_name="Produce",
    items=[
        GroceryItem(name="Bell Peppers", quantity="3 pieces", estimated_price="$3-4", category="Produce"),
        GroceryItem(name="Onions", quantity="2 lbs", estimated_price="$2-3", category="Produce")
    ],
    estimated_total="$5-7"
)
sample_section

ShoppingCategory(section_name='Produce', items=[GroceryItem(name='Bell Peppers', quantity='3 pieces', estimated_price='$3-4', category='Produce'), GroceryItem(name='Onions', quantity='2 lbs', estimated_price='$2-3', category='Produce')], estimated_total='$5-7')

In [62]:
print("\n🏪 Sample Shopping Section:")
display(JSON(sample_section.model_dump()))


🏪 Sample Shopping Section:


<IPython.core.display.JSON object>

##### `GroceryShoppingPlan`

It is the master plan that combines everything into a complete shopping strategy:

- **`total_budget`** – The overall spending limit for the shopping trip.
- **`meal_plans`** – Collection of meals being prepared (note this is a list).
- **`shopping_sections`** – Organized categories for store navigation (note this is a list).
- **`shopping_tips`** – Money-saving advice and practical suggestions (note this is a list).


In [63]:
class GroceryShoppingPlan(BaseModel):
    """Complete simplified shopping plan"""
    total_budget: str = Field(description="Total planned budget")
    meal_plans: List[MealPlan] = Field(description="Planned meals")
    shopping_sections: List[ShoppingCategory] = Field(description="Organized by store sections")
    shopping_tips: List[str] = Field(description="Money-saving and efficiency tips")

### Setting Up Our LLM and Essential Tools  

In [106]:
import os
from crewai_tools import SerperDevTool
from crewai import Agent, Task, Crew, Process
from crewai import LLM

llm = LLM(
    model="huggingface/Qwen/Qwen3.5-397B-A17B:novita",
    api_key=os.environ["HUGGINGFACEHUB_API_TOKEN"],
    base_url="https://router.huggingface.co/v1",
    max_retries=7,           # total retry attempts
    timeout=300,             # seconds per request
    retry_min_wait=4,        # initial backoff seconds
    retry_max_wait=35
)

##### Meal and Grocery Planning Workflow Overview

![image (3).png](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/0crjoLc3Yn0JefQoJVUarA/image%20-3-.png)


##### Creating Our AI Agent Workflow with CrewAI

We use CrewAI to build a collaborative meal-planning workflow with specialized agents. Our first agent is a **Meal Planner & Recipe Researcher** that:

- has a clear role/goal to find and design strong recipe plans,  
- uses a backstory to guide domain-specific behavior (dietary needs, skill level),  
- is equipped with `SerperDevTool()` for real-time recipe and ingredient research,  
- runs with `verbose=False` to keep output clean.

In [108]:
meal_planner = Agent(
    role="Meal Planner & Recipe Researcher",
    goal="Search for optimal recipes and create detailed meal plans",
    backstory=(
        "A skilled meal planner who researches the best recipes online, considering dietary needs, cooking skill levels, and budget constraints."
        "Return structured data only."
    ),
    tools=[SerperDevTool()],
    verbose=True,
    llm = llm,
    allow_delegation=True
)
meal_planner

Agent(role=Meal Planner & Recipe Researcher, goal=Search for optimal recipes and create detailed meal plans, backstory=A skilled meal planner who researches the best recipes online, considering dietary needs, cooking skill levels, and budget constraints.Return structured data only.)

##### Defining Our Meal Planning Task

In CrewAI, a task is a clear assignment that defines what an agent should do, what inputs it gets, and what output it must produce.  
Here, we define a reusable meal-planning task with dynamic placeholders (`{meal_name}`, `{servings}`, `{budget}`, `{dietary_restrictions}`, `{cooking_skill}`) and standardized output, optionally saved to `meals.json`.

In [109]:
import json

meal_planning_task = Task(
    description=(
        "Search for the best '{meal_name}' recipe for {servings} people within a {budget} budget. "
        "Consider dietary restrictions: {dietary_restrictions} and cooking skill level: {cooking_skill}. "
        "Find recipes that match the skill level and provide complete ingredient lists with quantities."
    ),
    expected_output="A detailed meal plan with researched ingredients, quantities, and cooking instructions appropriate for the skill level.",
    agent=meal_planner,
    # callback=process_pydantic(MealPlan),
    output_pydantic=MealPlan,
    output_file="meals.json"
)
meal_planning_task

Task(description=Search for the best '{meal_name}' recipe for {servings} people within a {budget} budget. Consider dietary restrictions: {dietary_restrictions} and cooking skill level: {cooking_skill}. Find recipes that match the skill level and provide complete ingredient lists with quantities., expected_output=A detailed meal plan with researched ingredients, quantities, and cooking instructions appropriate for the skill level.)

##### Creating and Running Our Meal Planning Crew

In CrewAI, a Crew coordinates agents and their tasks like a specialist team working through a workflow.  
Here, we run a single-agent crew in sequential mode to research and generate structured meal insights for the given recipe requirements, then store results. This is a partial output that will later be expanded with the full Pydantic models for a complete shopping plan.

In [70]:
from crewai import Crew, Process

meal_planner_crew = Crew(
    agents=[meal_planner],
    tasks=[meal_planning_task],
    process=Process.sequential,  # Ensures tasks are executed in order
    verbose=True
)

In [101]:
meal_planner_result = meal_planner_crew.kickoff(
    inputs={
        "meal_name": "Chicken Stir Fry",
        "servings": 4,
        "budget": "$25",                           
        "dietary_restrictions": ["no nuts"],       
        "cooking_skill": "beginner"                
    }
)
print("✅ Single meal planning completed!")
print("📋 Single Meal Results:")
print(meal_planner_result)

╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: d5e66db3-014e-4e3a-89f8-e9d1316783fa                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Meal Planner & Recipe Researcher                                                                        │
│                                                                                                                 │
│  Task: Search for the best 'Chicken Stir Fry' recipe for 4 people within a $25 budget. Consider dietary         │
│  restrictions: ['no nuts'] and cooking skill level: beginner. Find recipes that match the skill level and       │
│  provide complete ingredient lists with quantities.                                                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

╭──────────────────────────────────────────── 🔧 Agent Tool Execution ────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Meal Planner & Recipe Researcher                                                                        │
│                                                                                                                 │
│  Thought: Thought: I need to search for a beginner-friendly Chicken Stir Fry recipe for 4 people under $25,     │
│  ensuring no nuts are included. I'll use the search tool to find suitable recipes with detailed ingredient      │
│  lists.                                                                                                         │
│                                                                                                                 │
│  Using Tool: Search the internet with Serper                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Input ───────────────────────────────────────────────────╮
│                                                                                                                 │
│  {                                                                                                              │
│    "search_query": "best chicken stir fry recipe for 4 people under $25 budget no nuts beginner friendly"       │
│  }                                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

✅ Single meal planning completed!

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: d5e66db3-014e-4e3a-89f8-e9d1316783fa                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: {"meal_name": "Easy Nut-Free Chicken Stir Fry", "difficulty_level": "Easy", "servings": 4,       │
│  "researched_ingredients": ["1 lb chicken breast, sliced", "2 cups broccoli florets", "1 red bell pepper,       │
│  sliced", "1 carrot, julienned", "2 cloves garlic, minced", "1 tbsp fresh ginger, grated", "3 tbsp soy sauce    │
│  (or tamari for gluten-free)", "1 tbsp sesame oil", "2 cups cooked rice", "1 tbsp vegetable oil for cooking"]}  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


📋 Single Meal Results:


meal_name='Easy Nut-Free Chicken Stir Fry' difficulty_level='Easy' servings=4 researched_ingredients=['1 lb chicken breast, sliced', '2 cups broccoli florets', '1 red bell pepper, sliced', '1 carrot, julienned', '2 cloves garlic, minced', '1 tbsp fresh ginger, grated', '3 tbsp soy sauce (or tamari for gluten-free)', '1 tbsp sesame oil', '2 cups cooked rice', '1 tbsp vegetable oil for cooking']


╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [86]:
meal_planner_result.pydantic

MealPlan(meal_name='Easy Beginner Chicken Stir Fry', difficulty_level='Easy', servings=4, researched_ingredients=['1.5 lbs boneless skinless chicken breasts, cut into bite-sized pieces', '3 tablespoons low sodium soy sauce', '3 tablespoons brown sugar', '1 tablespoon toasted sesame oil', '3 cloves garlic, minced', '1 tablespoon fresh ginger, grated', '1 tablespoon cornstarch', '2 tablespoons cold water', '2 tablespoons vegetable or canola oil for cooking', '2 cups broccoli florets', '1 red bell pepper, sliced', '1 cup snap peas or snow peas', '1 cup carrots, sliced into matchsticks', '1/4 teaspoon black pepper', 'Optional: 1 teaspoon sriracha for mild heat', 'Cooked rice for serving (2 cups uncooked rice)'])

### Creating Our Shopping Organization Agent

After meal planning, we add a **Shopping Organizer** agent to convert meal plans into efficient, store-ready shopping lists.  
It’s separate for specialization: it focuses on store layout, quantity calculation, and budget-aware organization (produce/dairy/meat/pantry), and works entirely from internal meal-plan data without external tools.

In [111]:
shopping_organizer = Agent(
    role="Shopping Organizer", 
    goal="Organize grocery lists by store sections efficiently",
    backstory=(
        "An experienced shopper who knows how to organize lists for quick store trips and considers dietary restrictions."
        "Return structured data only."
    ),
    tools=[],
    llm=llm,
    verbose=False,
    allow_delegation=True
)

##### Defining the Shopping Organization Task

This task converts the meal planner’s output into a practical shopping list.  
It uses `context=[meal_planning_task]` to consume prior results, `output_pydantic=GroceryShoppingPlan` for structured validation, organizes items by store section, and tracks budget constraints.  
Context chaining is what lets agents build seamlessly on each other’s work.

In [112]:
shopping_task = Task(
    description=(
        "Organize the ingredients from the '{meal_name}' meal plan into a grocery shopping list. "
        "Group items by store sections and estimate quantities for {servings} people. "
        "Consider dietary restrictions: {dietary_restrictions} and cooking skill: {cooking_skill}. "
        "Stay within budget: {budget}."
    ),
    expected_output="An organized shopping list grouped by store sections with quantities and prices.",
    agent=shopping_organizer,
    context=[meal_planning_task],
    output_pydantic=GroceryShoppingPlan,
    output_file="shopping_list.json"
)

##### Building Our Two-Agent Grocery Crew

This multi-agent crew creates a full meal-to-shopping-list pipeline: meal planner first, then shopping organizer, using `Process.sequential`.  
Task context passes meal-planning output downstream automatically, and final output is structured with `GroceryShoppingPlan`.  
Benefits: clear role separation, reusable workflow components, and reliable structured results.

In [83]:
two_agent_grocery_crew = Crew(
    agents=[meal_planner, shopping_organizer],  # Both agents
    tasks=[meal_planning_task, shopping_task],   # Both tasks
    process=Process.sequential,
    verbose=True
)

# Run the complete crew (this will do BOTH meal planning AND shopping)
shopping_result = two_agent_grocery_crew.kickoff(
    inputs={
        "meal_name": "Chicken Stir Fry",
        "servings": 4,
        "budget": "$25",                           
        "dietary_restrictions": ["no nuts"],      
        "cooking_skill": "beginner"               
    }
)

# Print the shopping results
print("✅ Complete meal planning + shopping completed!")
print("📋 Shopping Results:")
print(shopping_result)

╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: f1d7c25c-0916-4067-a44c-f6b3fd20ebe3                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Meal Planner & Recipe Researcher                                                                        │
│                                                                                                                 │
│  Task: Search for the best 'Chicken Stir Fry' recipe for 4 people within a $25 budget. Consider dietary         │
│  restrictions: ['no nuts'] and cooking skill level: beginner. Find recipes that match the skill level and       │
│  provide complete ingredient lists with quantities.                                                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

╭──────────────────────────────────────────── 🔧 Agent Tool Execution ────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Meal Planner & Recipe Researcher                                                                        │
│                                                                                                                 │
│  Thought: Thought: I need to search for Chicken Stir Fry recipes that match the criteria: 4 servings, $25       │
│  budget, no nuts, beginner-friendly. Let me start by searching for suitable recipes.                            │
│                                                                                                                 │
│  Using Tool: Search the internet with Serper                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Shopping Organizer                                                                                      │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {                                                                                                              │
│    "total_budget": "$25.00",                                                                                    │
│    "meal_plans": [                                                                                              │
│      {                                                                                                          │
│        "meal_name": "Easy Chicken Stir Fry with Vegetables",                                                    │
│        "difficulty_level": "Easy",                                                                              │
│        "servings": 4,                                                                                           │
│        "researched_ingredients": [                                                                              │
│          "1 lb boneless, skinless chicken breast, cut into 1-inch cubes",                                       │
│          "2 tablespoons olive oil or vegetable oil",                                                            │
│          "2 cups broccoli florets",                                                                             │
│          "1 cup bell peppers (red or yellow), sliced",                                                          │
│          "1 cup snap peas or green beans",                                                                      │
│          "1 cup carrots, sliced into thin rounds",                                                              │
│          "3 cloves garlic, minced",                                                                             │
│          "1 tablespoon fresh ginger, grated",                                                                   │
│          "3 tablespoons low-sodium soy sauce",                                                                  │
│          "1 tablespoon honey or brown sugar",                                                                   │
│          "1 teaspoon sesame oil",                                                                               │
│          "1 tablespoon cornstarch",                                                                             │
│          "1/4 cup water or chicken broth",                                                                      │
│          "Salt and pepper to taste",                                                                            │
│          "Optional: 1/2 teaspoon sriracha or red pepper flakes for heat",                                       │
│          "Cooked rice or noodles for serving (optional)"                                                        │
│        ]                                                                                                        │
│      }                                                                                                          │
│    ],                                                                                                           │
│    "shopping_sections": [                                                                                       │
│      {                                                                                                          │
│        "section_name": "Meat Department",              

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 596bb164-19ef-495d-9a28-3c13779c3e54                                                                     │
│  Agent: Shopping Organizer                                                                                      │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

✅ Complete meal planning + shopping completed!
📋 Shopping Results:
total_budget='$25.00' meal_plans=[MealPlan(meal_name='Easy Chicken Stir Fry with Vegetables', difficulty_level='Easy', servings=4, researched_ingredients=['1 lb boneless, skinless chicken breast, cut into 1-inch cubes', '2 tablespoons olive oil or vegetable oil', '2 cups broccoli florets', '1 cup bell peppers (red or yellow), sliced', '1 cup snap peas or green beans', '1 cup carrots, sliced into thin rounds', '3 cloves garlic, minced', '1 tablespoon fresh ginger, grated', '3 tablespoons low-sodium soy sauce', '1 tablespoon honey or brown sugar', '1 teaspoon sesame oil', '1 tablespoon cornstarch', '1/4 cup water or chicken broth', 'Salt and pepper to taste', 'Optional: 1/2 teaspoon sriracha or red pepper flakes for heat', 'Cooked rice or noodles for serving (optional)'])] shopping_sections=[ShoppingCategory(section_name='Meat Department', items=[GroceryItem(name='Chicken Breast (Boneless, Skinless)', quantity='1 lb', est

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: f1d7c25c-0916-4067-a44c-f6b3fd20ebe3                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: {                                                                                                │
│    "total_budget": "$25.00",                                                                                    │
│    "meal_plans": [                                                                                              │
│      {                                                                                                          │
│        "meal_name": "Easy Chicken Stir Fry with Vegetables",                                                    │
│        "difficulty_level": "Easy",                                                                              │
│        "servings": 4,                                                                                           │
│        "researched_ingredients": [                                                                              │
│          "1 lb boneless, skinless chicken breast, cut into 1-inch cubes",                                       │
│          "2 tablespoons olive oil or vegetable oil",                                                            │
│          "2 cups broccoli florets",                                                                             │
│          "1 cup bell peppers (red or yellow), sliced",                                                          │
│          "1 cup snap peas or green beans",                                                                      │
│          "1 cup carrots, sliced into thin rounds",                                                              │
│          "3 cloves garlic, minced",                                                                             │
│          "1 tablespoon fresh ginger, grated",                                                                   │
│          "3 tablespoons low-sodium soy sauce",                                                                  │
│          "1 tablespoon honey or brown sugar",                                                                   │
│          "1 teaspoon sesame oil",                                                                               │
│          "1 tablespoon cornstarch",                                                                             │
│          "1/4 cup water or chicken broth",                                                                      │
│          "Salt and pepper to taste",                                                                            │
│          "Optional: 1/2 teaspoon sriracha or red pepper flakes for heat",                                       │
│          "Cooked rice or noodles for serving (optional)"                                                        │
│        ]                                                                                                        │
│      }                                                                                                          │
│    ],                                                                                                           │
│    "shopping_sections": [                                                                                       │
│      {                                                

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [85]:
shopping_result.pydantic

GroceryShoppingPlan(total_budget='$25.00', meal_plans=[MealPlan(meal_name='Easy Chicken Stir Fry with Vegetables', difficulty_level='Easy', servings=4, researched_ingredients=['1 lb boneless, skinless chicken breast, cut into 1-inch cubes', '2 tablespoons olive oil or vegetable oil', '2 cups broccoli florets', '1 cup bell peppers (red or yellow), sliced', '1 cup snap peas or green beans', '1 cup carrots, sliced into thin rounds', '3 cloves garlic, minced', '1 tablespoon fresh ginger, grated', '3 tablespoons low-sodium soy sauce', '1 tablespoon honey or brown sugar', '1 teaspoon sesame oil', '1 tablespoon cornstarch', '1/4 cup water or chicken broth', 'Salt and pepper to taste', 'Optional: 1/2 teaspoon sriracha or red pepper flakes for heat', 'Cooked rice or noodles for serving (optional)'])], shopping_sections=[ShoppingCategory(section_name='Meat Department', items=[GroceryItem(name='Chicken Breast (Boneless, Skinless)', quantity='1 lb', estimated_price='$5.75', category='Meat')], esti

##### Adding Financial Intelligence with a Budget Advisor Agent

To complement meal planning and shopping organization, add a **Budget Advisor** focused on cost control.  
It uses web search (`SerperDevTool`) to check current prices/deals, suggests lower-cost substitutions, and provides actionable savings tips using context from earlier workflow steps.

In [113]:
budget_advisor = Agent(
    role="Budget Advisor",
    goal="Provide cost estimates and money-saving tips",
    backstory=(
        "A budget-conscious shopper who helps families save money on groceries while respecting dietary needs."
    ),
    tools=[SerperDevTool()],
    llm=llm,
    verbose=False,
    allow_delegation=True
)

##### Defining the Budget Analysis Task

This task acts as a financial checkpoint by taking the full shopping plan and adding cost analysis, budget validation, and savings recommendations.  
It uses `context=[meal_planning_task, shopping_task]`, combines prior outputs with current web price checks, and suggests cheaper alternatives plus practical money-saving tips.

In [114]:
budget_task = Task(
    description=(
        "Analyze the shopping plan for '{meal_name}' serving {servings} people. "
        "Ensure total cost stays within {budget}. Consider dietary restrictions: {dietary_restrictions}. "
        "Provide practical money-saving tips and alternative ingredients if needed to meet budget."
    ),
    expected_output="A complete shopping guide with detailed prices, budget analysis, and money-saving tips.",
    agent=budget_advisor,
    context=[meal_planning_task, shopping_task],
    output_file="shopping_guide.md"
)

### Using YAML with CrewAI (Food Leftover Agent and Task)

**YAML** is a simple, human-readable config format that supports nested structures and keeps configuration separate from code.

With CrewAI, YAML lets you define agents, tasks, and crew execution flow (`sequential`, `hierarchical`) without changing Python logic.  
This makes iteration faster, collaboration easier (including non-developers), and workflows more production-ready.

You can define:
- **Agents**: role, goal, backstory, tools, verbosity  
- **Tasks**: description, expected output, assigned agent, dependencies  
- **Crew**: process/orchestration settings  

(Example files: `agents.yaml`, `tasks.yaml`)

In [90]:
!wget "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/-5NXnHGkX1WG81tqE8kSTg/agents.yaml"

--2026-03-15 10:01:07--  https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/-5NXnHGkX1WG81tqE8kSTg/agents.yaml
Resolving cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)... 169.45.118.108
Connecting to cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)|169.45.118.108|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 337 [text/x-yaml]
Saving to: ‘agents.yaml’

agents.yaml         100%[===================>]     337  --.-KB/s    in 0s      

2026-03-15 10:01:09 (64.3 MB/s) - ‘agents.yaml’ saved [337/337]



In [91]:
!wget "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/7fhq1KPxi7cuqjOCBgcmXQ/tasks.yaml"

--2026-03-15 10:01:14--  https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/7fhq1KPxi7cuqjOCBgcmXQ/tasks.yaml
Resolving cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)... 169.45.118.108
Connecting to cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)|169.45.118.108|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 517 [text/x-yaml]
Saving to: ‘tasks.yaml’

tasks.yaml          100%[===================>]     517  --.-KB/s    in 0s      

2026-03-15 10:01:16 (61.6 MB/s) - ‘tasks.yaml’ saved [517/517]



In [92]:
%mkdir -p config
%mv agents.yaml  config/agents.yaml
%mv tasks.yaml   config/tasks.yaml

##### Using CrewBase and Decorators with CrewAI

`@CrewBase` marks a class as the container for your CrewAI workflow and lets CrewAI automatically discover and wire agents/tasks (and crew hooks), especially useful in Python, YAML, or hybrid setups.

Why use it:
- Auto-discovers `@agent`, `@task`, and optional `@crew` methods
- Auto-loads YAML config when available
- Keeps larger projects modular, clean, and scalable

| Decorator           | Marks…                                            |
| :------------------ | :----------------------------------------------- |
| `@CrewBase`         | The class that holds your agents & tasks         |
| `@agent`            | A method that returns an `Agent` object          |
| `@task`             | A method that returns a `Task` object            |
| `@crew`             | (Optional) A method that returns a `Crew` object |
| `@before_kickoff`   | (Optional) Runs once **before** the crew starts  |
| `@after_kickoff`    | (Optional) Runs once **after** the crew finishes |

`@CrewBase` + decorators reduce manual wiring and make workflow management easier as your app grows.

##### How `@CrewBase` Works with YAML in This Project

This project uses a hybrid setup: Python methods + YAML configs.  
- `@agent` loads from `self.agents_config["leftover_manager"]` and returns an `Agent`.  
- `@task` loads from `self.tasks_config["leftover_task"]` and returns a `Task` tied to that agent.  

In the notebook, importing `LeftoverCrew` gives you these dynamically built objects.

`agents.yaml` and `tasks.yaml` must be under `config/` so `@CrewBase` can auto-discover them.

We also moved the crew class to `leftover.py` because `@CrewBase` relies on Python file inspection (`inspect.getfile(cls)`), which doesn’t work reliably for classes defined directly in notebook cells.

In [115]:
from leftover import LeftoversCrew

leftovers_cb = LeftoversCrew(llm=llm)
yaml_leftover_manager = leftovers_cb.leftover_manager()
yaml_leftover_task    = leftovers_cb.leftover_task()

> 💡 **Note:** While `@CrewBase` makes your agent/task management easier, it wraps your class behind the scenes.
>
> That means:
> - You can't always use `__init__()` or pass custom objects like in normal Python classes.
> - Testing and debugging `@CrewBase` classes inside Jupyter notebooks can be tricky due to how Jupyter handles decorators and dynamic reloading.
>
> 📁 For stability, define your `@CrewBase` classes in separate `.py` files instead of notebooks. This makes CrewAI's configuration loading and agent resolution work reliably.
>
> 🔍 See the [CrewAI documentation](https://docs.crewai.com/) for full details on how `@CrewBase` works.

##### Defining Our Summary Agent and Task

To manage all the content generated by the agents, we will now create one final agent and its corresponding task which will gather all the content and create a detailed summary. 


In [116]:
summary_agent = Agent(
    role="Report Compiler",
    goal="Compile comprehensive meal planning reports from all team outputs",
    backstory="A skilled coordinator who organizes information from multiple specialists into comprehensive, easy-to-follow reports.",
    tools=[],
    llm=llm,
    verbose=False
)

In [117]:
summary_task = Task(
    description=(
        "Compile a comprehensive meal planning report that includes:\n"
        "1. The complete recipe and cooking instructions from the meal planner\n"
        "2. The organized shopping list with prices from the shopping organizer\n"
        "3. The budget analysis and money-saving tips from the budget advisor\n"
        "4. The leftover management suggestions from the waste reduction specialist\n"
        "Format this as a complete, user-friendly meal planning guide."
    ),
    expected_output="A comprehensive meal planning guide that combines all team outputs into one cohesive report.",
    agent=summary_agent,
    context=[meal_planning_task, shopping_task, budget_task, yaml_leftover_task],
)

##### Assembling Our Complete Grocery Planning Team

Now we combine five specialists into one end-to-end grocery planning crew, from meal idea to waste-aware shopping guide:

1. **Meal Planner**: researches recipes and creates meal plans  
2. **Shopping Organizer**: converts plans into store-organized lists  
3. **Budget Advisor**: adds cost analysis and savings strategies  
4. **Leftover Manager**: suggests ways to use leftovers and reduce waste  
5. **Report Compiler**: merges all outputs into one clear final guide

In [98]:
complete_grocery_crew = Crew(
    agents=[
        meal_planner,           
        shopping_organizer,      
        budget_advisor,         
        yaml_leftover_manager,  # YAML-based leftover manager
        summary_agent           # New summary agent
    ],
    tasks=[
        meal_planning_task,     
        shopping_task,          
        budget_task,            
        yaml_leftover_task,    # YAML-based leftover task
        summary_task            # New summary task
    ],
    process=Process.sequential,
    verbose=True
)

##### Executing Our Complete Grocery Planning Workflow

Run the five-agent workflow with detailed inputs (`Chicken Stir Fry`, 4 servings, `$25` budget, `["no nuts", "low sodium"]`, `beginner`) to showcase full capability.

Execution flow:
1. Meal Planner finds suitable recipes  
2. Shopping Organizer builds store-section shopping lists  
3. Budget Advisor checks costs and suggests savings  
4. Leftover Manager proposes ways to reuse extras and reduce waste  
5. Report Compiler combines everything into one clear, user-friendly guide  

Result: a complete grocery plan tailored to dietary, skill, and budget constraints.

In [99]:
# Run the complete crew
complete_result = complete_grocery_crew.kickoff(
    inputs={
        "meal_name": "Chicken Stir Fry",
        "servings": 4,
        "budget": "$25",
        "dietary_restrictions": ["no nuts", "low sodium"],
        "cooking_skill": "beginner"
    }
)

print("✅ Complete meal planning with summary completed!")
print("📋 Complete Results:")
print(complete_result)

╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 34ed8fb0-d488-4db6-a4f7-5752e7e57815                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Meal Planner & Recipe Researcher                                                                        │
│                                                                                                                 │
│  Task: Search for the best 'Chicken Stir Fry' recipe for 4 people within a $25 budget. Consider dietary         │
│  restrictions: ['no nuts', 'low sodium'] and cooking skill level: beginner. Find recipes that match the skill   │
│  level and provide complete ingredient lists with quantities.                                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

Output()

╭──────────────────────────────────────────── 🔧 Agent Tool Execution ────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Meal Planner & Recipe Researcher                                                                        │
│                                                                                                                 │
│  Thought: Thought: I need to search for Chicken Stir Fry recipes that meet the specific requirements: 4         │
│  servings, $25 budget, no nuts, low sodium, and beginner-friendly. Let me start by searching for suitable       │
│  recipes online.                                                                                                │
│                                                                                                                 │
│  Using Tool: Search the internet with Serper                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: leftover_task                                                                                            │
│  Agent: Leftover & Waste Reduction Specialist                                                                   │
│                                                                                                                 │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Report Compiler                                                                                         │
│                                                                                                                 │
│  Task: Compile a comprehensive meal planning report that includes:                                              │
│  1. The complete recipe and cooking instructions from the meal planner                                          │
│  2. The organized shopping list with prices from the shopping organizer                                         │
│  3. The budget analysis and money-saving tips from the budget advisor                                           │
│  4. The leftover management suggestions from the waste reduction specialist                                     │
│  Format this as a complete, user-friendly meal planning guide.                                                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: b8267e6d-7922-47c7-8ec9-a3aa01d07ad2                                                                     │
│  Agent: Report Compiler                                                                                         │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [102]:
print(complete_result)

# 🍽️ COMPLETE MEAL PLANNING GUIDE
## Easy Low-Sodium Chicken Stir Fry (Serves 4)

---

# PART 1: RECIPE & COOKING INSTRUCTIONS

## 📋 Recipe Overview

| Detail | Information |
|--------|-------------|
| **Meal Name** | Easy Low-Sodium Chicken Stir Fry |
| **Difficulty Level** | Easy |
| **Servings** | 4 people |
| **Prep Time** | 15 minutes |
| **Cook Time** | 15 minutes |
| **Total Time** | 30 minutes |
| **Dietary Notes** | ✅ Low Sodium, ✅ Nut-Free |

---

## 🛒 Complete Ingredients List

### Main Ingredients
| Ingredient | Quantity | Notes |
|------------|----------|-------|
| Boneless, skinless chicken breasts | 1 lb (450g) | Cut into bite-sized pieces |
| Vegetable oil or olive oil | 2 tablespoons | For cooking |
| Broccoli florets | 1 cup | Fresh or frozen |
| Bell peppers (red, yellow, or green) | 1 cup | Sliced |
| Snap peas or snow peas | 1 cup | Fresh |
| Carrot | 1 medium | Sliced into thin strips |
| Onion | ½ medium | Sliced |
| Garlic | 3 cloves | Minced |
| Fresh ginger | 

### 📝 Exercise 1 - Create a Specialized Dietary Agent

**Objective:** 

Add a fourth agent that specializes in dietary analysis and nutritional recommendations.

**Create a new agent called nutrition_analyst that:**
* Analyzes the nutritional content of meal plans.
* Suggests healthier alternatives when needed.
* Provides calorie and macronutrient estimates.
* Considers specific dietary goals (weight loss, muscle building, etc).

**Requirements:**
1. Create the agent with appropriate role, goal, and backstory
2. Define a task that analyzes the meal plan for nutritional value
3. Add it to your crew workflow
4. Test it with a meal request that includes nutritional goals


In [118]:
# Your code here
nutrition_analyst = Agent(
    role="Nutrition Analyst & Health Advisor",
    goal="Analyze meal nutritional content and provide healthy recommendations",
    backstory="A certified nutritionist who evaluates meals for nutritional balance, calorie content, and health optimization while respecting dietary restrictions.",
    tools=[SerperDevTool()],
    llm=llm,
    verbose=False
)

nutrition_task = Task(
    description=(
        "Analyze the nutritional content of the '{meal_name}' meal plan for {servings} people. "
        "Calculate approximate calories, protein, carbs, and fats. Consider dietary restrictions: {dietary_restrictions}. "
        "Provide healthy alternatives if the meal could be more nutritious while staying within {budget}."
    ),
    expected_output="Nutritional analysis with calorie estimates, macronutrient breakdown, and healthy improvement suggestions.",
    agent=nutrition_analyst,
    context=[meal_planning_task, shopping_task, budget_task],
    output_file="nutrition_analysis.md"
)

# Test with expanded crew
health_focused_crew = Crew(
    agents=[meal_planner, shopping_organizer, budget_advisor, nutrition_analyst, yaml_leftover_manager, summary_agent],
    tasks=[meal_planning_task, shopping_task, budget_task, nutrition_task, yaml_leftover_task, summary_task],
    process=Process.sequential,
    verbose=True
)

nutrition_research_result = health_focused_crew.kickoff(
    inputs={
        "meal_name": "Quinoa Buddha Bowl",
        "servings": 2,
        "budget": "$20",
        "dietary_restrictions": ["vegetarian", "high protein"],
        "cooking_skill": "intermediate"
    }
)

print("✅ Nutritional Research completed!")
print("📋 Nutritional Research Results:")
print(nutrition_research_result)

╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: aa52de75-9e90-42f9-b002-fbf9c4ef6496                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Meal Planner & Recipe Researcher                                                                        │
│                                                                                                                 │
│  Task: Search for the best 'Quinoa Buddha Bowl' recipe for 2 people within a $20 budget. Consider dietary       │
│  restrictions: ['vegetarian', 'high protein'] and cooking skill level: intermediate. Find recipes that match    │
│  the skill level and provide complete ingredient lists with quantities.                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

Output()

╭──────────────────────────────────────────── 🔧 Agent Tool Execution ────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Meal Planner & Recipe Researcher                                                                        │
│                                                                                                                 │
│  Thought: Thought: I need to search for Quinoa Buddha Bowl recipes that meet the specific criteria:             │
│  vegetarian, high protein, for 2 people, within $20 budget, and intermediate skill level. I'll start by         │
│  searching online for the best recipes.                                                                         │
│                                                                                                                 │
│  Using Tool: Search the internet with Serper                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Report Compiler                                                                                         │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # 🥗 COMPLETE MEAL PLANNING GUIDE                                                                              │
│  ## Quinoa Chickpea Buddha Bowl (2 Servings)                                                                    │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  # PART 1: RECIPE & COOKING INSTRUCTIONS                                                                        │
│                                                                                                                 │
│  ## Recipe Overview                                                                                             │
│  | Detail | Information |                                                                                       │
│  |--------|-------------|                                                                                       │
│  | **Meal Name** | Quinoa Chickpea Buddha Bowl |                                                                │
│  | **Difficulty Level** | Medium |                                                                              │
│  | **Servings** | 2 |                                                                                           │
│  | **Prep Time** | 15 minutes |                                                                                 │
│  | **Cook Time** | 25-30 minutes |                                                                              │
│  | **Total Time** | 40-45 minutes |                                                                             │
│  | **Dietary** | ✓ Vegetarian ✓ High Protein ✓ Gluten-Free |                                                    │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## Ingredients List                                                                                            │
│                                                                                                                 │
│  ### For the Bowl:                                                                                              │
│  | Ingredient | Quantity | Notes |                                                                              │
│  |------------|----------|-------|                                                                              │
│  | Dry chickpeas | 1/2 cup (or 1 can, 15 oz, drained) | Soak overnight if using dry |                           │
│  | White quinoa | 1/2 cup | Rinse before cooking |                                                              │
│  | Curly/dinosaur kale | 1/2 large bunch (4-5 cups chopped) | Remove tough stems |                              │
│  | Olive oil | 2 tablespoons (divided) | For roasting and dressing |                                            │
│  | Garlic powder | 1/2 teaspoon (divided) | Divide betwe

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 817243b6-d8c9-427f-934b-e46b43bef4e5                                                                     │
│  Agent: Report Compiler                                                                                         │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

✅ Nutritional Research completed!
📋 Nutritional Research Results:
# 🥗 COMPLETE MEAL PLANNING GUIDE
## Quinoa Chickpea Buddha Bowl (2 Servings)

---

# PART 1: RECIPE & COOKING INSTRUCTIONS

## Recipe Overview
| Detail | Information |
|--------|-------------|
| **Meal Name** | Quinoa Chickpea Buddha Bowl |
| **Difficulty Level** | Medium |
| **Servings** | 2 |
| **Prep Time** | 15 minutes |
| **Cook Time** | 25-30 minutes |
| **Total Time** | 40-45 minutes |
| **Dietary** | ✓ Vegetarian ✓ High Protein ✓ Gluten-Free |

---

## Ingredients List

### For the Bowl:
| Ingredient | Quantity | Notes |
|------------|----------|-------|
| Dry chickpeas | 1/2 cup (or 1 can, 15 oz, drained) | Soak overnight if using dry |
| White quinoa | 1/2 cup | Rinse before cooking |
| Curly/dinosaur kale | 1/2 large bunch (4-5 cups chopped) | Remove tough stems |
| Olive oil | 2 tablespoons (divided) | For roasting and dressing |
| Garlic powder | 1/2 teaspoon (divided) | Divide between chickpeas and dressing

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: aa52de75-9e90-42f9-b002-fbf9c4ef6496                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: # 🥗 COMPLETE MEAL PLANNING GUIDE                                                                │
│  ## Quinoa Chickpea Buddha Bowl (2 Servings)                                                                    │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  # PART 1: RECIPE & COOKING INSTRUCTIONS                                                                        │
│                                                                                                                 │
│  ## Recipe Overview                                                                                             │
│  | Detail | Information |                                                                                       │
│  |--------|-------------|                                                                                       │
│  | **Meal Name** | Quinoa Chickpea Buddha Bowl |                                                                │
│  | **Difficulty Level** | Medium |                                                                              │
│  | **Servings** | 2 |                                                                                           │
│  | **Prep Time** | 15 minutes |                                                                                 │
│  | **Cook Time** | 25-30 minutes |                                                                              │
│  | **Total Time** | 40-45 minutes |                                                                             │
│  | **Dietary** | ✓ Vegetarian ✓ High Protein ✓ Gluten-Free |                                                    │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## Ingredients List                                                                                            │
│                                                                                                                 │
│  ### For the Bowl:                                                                                              │
│  | Ingredient | Quantity | Notes |                                                                              │
│  |------------|----------|-------|                                                                              │
│  | Dry chickpeas | 1/2 cup (or 1 can, 15 oz, drained) | Soak overnight if using dry |                           │
│  | White quinoa | 1/2 cup | Rinse before cooking |                                                              │
│  | Curly/dinosaur kale | 1/2 large bunch (4-5 cups chopped) | Remove tough stems |                              │
│  | Olive oil | 2 tablespoons (divided) | For roasting a

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

### 📝 Exercise 2 - Extend Pydantic Models for Weekly Planning

**Objective:**

Modify the existing Pydantic models to handle weekly meal planning instead of single meals.

**Create new Pydantic models that can handle:**

* Multiple meals across a week.
* Different meal types (breakfast, lunch, dinner).
* Weekly budget distribution.
* Bulk shopping optimization.

**Requirements:**

1. Create a WeeklyMealPlan model
2. Create a MealType enum (breakfast, lunch, dinner)
3. Modify GroceryShoppingPlan to handle weekly shopping
4. Test with a week's worth of meals


In [119]:
from enum import Enum
from typing import Dict

class MealType(str, Enum):
    BREAKFAST = "breakfast"
    LUNCH = "lunch" 
    DINNER = "dinner"
    SNACK = "snack"

class DailyMeals(BaseModel):
    """Meals for one day"""
    date: str = Field(description="Date in YYYY-MM-DD format")
    breakfast: Optional[MealPlan] = Field(default=None, description="Breakfast meal plan")
    lunch: Optional[MealPlan] = Field(default=None, description="Lunch meal plan") 
    dinner: Optional[MealPlan] = Field(default=None, description="Dinner meal plan")
    snacks: Optional[List[MealPlan]] = Field(default=None, description="Snack meal plans")
class WeeklyMealPlan(BaseModel):
    """Complete weekly meal planning"""
    week_start_date: str = Field(description="Start date of the week")
    daily_meals: List[DailyMeals] = Field(description="Meals for each day")
    weekly_themes: List[str] = Field(description="Cooking themes for the week")
    prep_suggestions: List[str] = Field(description="Meal prep recommendations")

class WeeklyGroceryPlan(BaseModel):
    """Weekly grocery shopping strategy"""
    weekly_budget: str = Field(description="Total weekly budget")
    meal_plans: List[DailyMeals] = Field(description="All weekly meals")
    shopping_sections: List[ShoppingCategory] = Field(description="Organized by store sections")
    bulk_items: List[GroceryItem] = Field(description="Items to buy in bulk")
    shopping_tips: List[str] = Field(description="Weekly shopping efficiency tips")
    budget_breakdown: Dict[str, str] = Field(description="Daily budget allocation")

# Test the models
sample_weekly_plan = WeeklyMealPlan(
    week_start_date="2024-01-15",
    daily_meals=[
        DailyMeals(
            date="2024-01-15",
            breakfast=MealPlan(meal_name="Oatmeal", difficulty_level="Easy", servings=2, researched_ingredients=["oats", "milk", "berries"]),
            lunch=MealPlan(meal_name="Salad", difficulty_level="Easy", servings=2, researched_ingredients=["lettuce", "tomatoes", "dressing"]),
            dinner=MealPlan(meal_name="Pasta", difficulty_level="Medium", servings=2, researched_ingredients=["pasta", "sauce", "cheese"])
        )
    ],
    weekly_themes=["Italian Monday", "Taco Tuesday"],
    prep_suggestions=["Wash vegetables on Sunday", "Cook grains in bulk"]
)

display(JSON(sample_weekly_plan.model_dump()))

<IPython.core.display.JSON object>